# NLI Models Script

In [1]:
from google.colab import drive
drive.mount('/content/drive/')

import os
nli_dir = '/content/drive/MyDrive/nli'
os.makedirs(nli_dir, exist_ok=True)
os.chdir(nli_dir)

print(os.getcwd())

Mounted at /content/drive/
/content/drive/MyDrive/nli


In [3]:
# Create dict of codes

import re
from config import I_1, I_2, I_3, I_4, I_5, I_NO_1, N_1, N_2, N_NO_1, J_1, J_2, J_3, J_NO_1

blocks = {
    "I_1": I_1, "I_2": I_2, "I_3": I_3, "I_4": I_4, "I_5": I_5, "I_NO_1": I_NO_1,
    "N_1": N_1, "N_2": N_2, "N_NO_1": N_NO_1,
    "J_1": J_1, "J_2": J_2, "J_3": J_3, "J_NO_1": J_NO_1,
}

CODE_PATTERN = re.compile(
    r'-\s*([A-Z0-9]+)\s*\n?\s*"(.*?)"\s*(?=\s*(?:\d+\.\d+\s+[A-Z0-9]+\s*)?-\s*[A-Z0-9]+|\s*\Z)',
    re.MULTILINE | re.DOTALL
)

codebook = {}
for name, block_text in blocks.items():
    for match in CODE_PATTERN.finditer(block_text):
        code_id, definition = match.groups()
        codebook[code_id] = definition.replace('\\"', '"').strip()

print(f"Parsed {len(codebook)} codes")

Parsed 127 codes


In [2]:
import pandas as pd
judaism = pd.read_csv("ucberkeley-dlab_target_jewish.csv")
code_features = pd.read_csv("code_features.csv").set_index("comment_id")
print(judaism.shape)
print(code_features.shape)

(1874, 3)
(1874, 381)


In [3]:
from transformers import pipeline
import torch

nli_pipe = pipeline(
    "zero-shot-classification",
    model="MoritzLaurer/deberta-v3-base-zeroshot-v1",
    device=0
)

print(nli_pipe.device)

config.json:   0%|          | 0.00/1.07k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  369MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/492 [00:00<?, ?B/s]

...

spm.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/8.65M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/23.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

cuda:0


In [6]:
# small test
test_code = "N1BELIEFS"
test_definition = codebook[test_code]
hypothesis = f"This text expresses: {test_definition}."

sample_texts = judaism["text"].sample(5, random_state=1).tolist()

for text in sample_texts:
    result = nli_pipe(text, candidate_labels=[test_definition], hypothesis_template="This text expresses: {}.")
    print(f"\nText: {text}")
    print(f"Entailment for {hypothesis}: {result['scores'][0]:.3f}")


Text: YTA. The worst part of this is gall at being offended by his "unprofessional" behavior. I'm gonna shit all over this person and then be surprised that it upset them. 🤣🤣🤣 Fucking classic sociopathic reaction. It's only business until someone reacts negatively to me then it's personal and they were the ones in the wrong. You a garbage sub-human.
Entailment for This text expresses: anti-Jewish beliefs, attitudes, actions or systemic conditions.: 0.163

Text: When will Democrats condemn all of the vile, anti-Semitic, anti-American rhetoric that Ilhan Omar and the rest of her "Squad" consistently spews? 🤔
Entailment for This text expresses: anti-Jewish beliefs, attitudes, actions or systemic conditions.: 0.998

Text: I'm Jewish and a granddaughter of holocaust survivors. Our Jewish/holocaust are the best (as long as we make them). My best friend gets a pass too
Entailment for This text expresses: anti-Jewish beliefs, attitudes, actions or systemic conditions.: 0.968

Text: However, m

In [ ]:
# Score full pilot corpus against all codes
import csv
import os
import torch
from tqdm.auto import tqdm

output_path = "nli_zeroshot_scores.csv"
BATCH_SIZE = 96  # tune if OOM on L4

texts = judaism["text"].tolist()
comment_ids = judaism["comment_id"].tolist()
code_ids = list(codebook.keys())
definitions = [codebook[c] for c in code_ids]

already_done = set()
if os.path.exists(output_path):
    existing = pd.read_csv(output_path)
    already_done = set(existing["code_id"].unique())
    print(f"Resuming: {len(already_done)} codes already scored, skipping those")

remaining = [(c, d) for c, d in zip(code_ids, definitions) if c not in already_done]

file_exists = os.path.exists(output_path)
with open(output_path, "a", newline="") as f:
    writer = csv.writer(f)
    if not file_exists:
        writer.writerow(["comment_id", "code_id", "entailment_score"])

    pbar = tqdm(remaining, desc="Scoring codes", unit="code")
    for code_id, definition in pbar:
        pbar.set_postfix(code=code_id)

        with torch.no_grad(), torch.autocast(device_type="cuda", dtype=torch.float16):
            outputs = nli_pipe(
                texts,
                candidate_labels=[definition],
                hypothesis_template="This text expresses: {}.",
                multi_label=True,
                batch_size=BATCH_SIZE,
                truncation=True,
            )

        if isinstance(outputs, dict):
            outputs = [outputs]
        all_scores = [o["scores"][0] for o in outputs]

        for cid, score in zip(comment_ids, all_scores):
            writer.writerow([cid, code_id, score])
        f.flush()

print("All codes scored.")

Resuming: 0 codes already scored, skipping those


Scoring codes:   0%|          | 0/127 [00:00<?, ?code/s]

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


All codes scored.


## V2

In [4]:
import csv
import os
import torch
import pandas as pd
from transformers import pipeline
from transformers.pipelines.pt_utils import KeyDataset
from datasets import Dataset
from tqdm.auto import tqdm

nli_pipe_v2 = pipeline(
    "zero-shot-classification",
    model="MoritzLaurer/deberta-v3-large-zeroshot-v2.0",
    device=0
)

config.json:   0%|          | 0.00/1.02k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  870MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.26k [00:00<?, ?B/s]

spm.model: reconstructing file:   0%|          |  0.00B / 2.46MB            

spm.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/8.66M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/23.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/970 [00:00<?, ?B/s]

In [8]:
# small test
test_code = "N1BELIEFS"
test_definition = codebook[test_code]
hypothesis = f"This text expresses: {test_definition}."

sample_texts = judaism["text"].sample(5, random_state=1).tolist()

for text in sample_texts:
    result = nli_pipe_v2(text, candidate_labels=[test_definition], hypothesis_template="This text expresses: {}.")
    print(f"\nText: {text}")
    print(f"Entailment for {hypothesis}: {result['scores'][0]:.3f}")


Text: YTA. The worst part of this is gall at being offended by his "unprofessional" behavior. I'm gonna shit all over this person and then be surprised that it upset them. 🤣🤣🤣 Fucking classic sociopathic reaction. It's only business until someone reacts negatively to me then it's personal and they were the ones in the wrong. You a garbage sub-human.
Entailment for This text expresses: anti-Jewish beliefs, attitudes, actions or systemic conditions.: 0.745

Text: When will Democrats condemn all of the vile, anti-Semitic, anti-American rhetoric that Ilhan Omar and the rest of her "Squad" consistently spews? 🤔
Entailment for This text expresses: anti-Jewish beliefs, attitudes, actions or systemic conditions.: 0.916

Text: I'm Jewish and a granddaughter of holocaust survivors. Our Jewish/holocaust are the best (as long as we make them). My best friend gets a pass too
Entailment for This text expresses: anti-Jewish beliefs, attitudes, actions or systemic conditions.: 0.935

Text: However, m

In [ ]:
# Score full pilot corpus against all codes, using a larger zero-shot model (v2)

output_path_v2 = "nli_zeroshot_scores_v2.csv"
BATCH_SIZE = 32  # tune if OOM on L4

texts = judaism["text"].tolist()
comment_ids = judaism["comment_id"].tolist()
code_ids = list(codebook.keys())
definitions = [codebook[c] for c in code_ids]
expected_n = len(comment_ids)

text_dataset = Dataset.from_dict({"text": texts})

# updated checkpoint
already_done = set()
if os.path.exists(output_path_v2):
    existing = pd.read_csv(output_path_v2)
    counts = existing["code_id"].value_counts()
    already_done = set(counts[counts >= expected_n].index)
    incomplete = set(counts[counts < expected_n].index)
    if incomplete:
        print(f"Found {len(incomplete)} incomplete codes, will be re-scored: {sorted(incomplete)}")
        existing = existing[~existing["code_id"].isin(incomplete)]
        existing.to_csv(output_path_v2, index=False)
    print(f"Resuming: {len(already_done)} codes fully scored, skipping those")

remaining = [(c, d) for c, d in zip(code_ids, definitions) if c not in already_done]

file_exists = os.path.exists(output_path_v2)
with open(output_path_v2, "a", newline="") as f:
    writer = csv.writer(f)
    if not file_exists:
        writer.writerow(["comment_id", "code_id", "entailment_score"])

    pbar = tqdm(remaining, desc="Scoring codes (v2)", unit="code")
    for code_id, definition in pbar:
        pbar.set_postfix(code=code_id)

        all_scores = []
        with torch.no_grad(), torch.autocast(device_type="cuda", dtype=torch.float16):
            for output in nli_pipe_v2(
                KeyDataset(text_dataset, "text"),
                candidate_labels=[definition],
                hypothesis_template="This text expresses: {}.",
                multi_label=True,
                batch_size=BATCH_SIZE,
                truncation=True,
            ):
                all_scores.append(output["scores"][0])

        if len(all_scores) != expected_n:
            print(f"WARNING: {code_id} produced {len(all_scores)} scores, expected {expected_n}. Skipping write, will retry next run.")
            continue

        for cid, score in zip(comment_ids, all_scores):
            writer.writerow([cid, code_id, score])
        f.flush()

print("All codes scored (v2).")

Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

Resuming: 12 codes fully scored, skipping those


Scoring codes (v2):   0%|          | 0/115 [00:00<?, ?code/s]

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


All codes scored (v2).


# Hypothesis Fix

The original NLI hypotheses reused `config.py`, which was built for LLM batch annotation, not standalone entailment scoring. Several code definitions were list fragments or normative carve-out clauses that only made sense in that context, producing ungrammatical or meaningless hypotheses when passed to NLI directly (e.g. "This text expresses: cemeteries.").

I introduce `nli_config.py`, which reconstructs each code's definition as a grammatically complete, standalone hypothesis, applying only minimal edits to the original source text (restoring a dropped subject or antecedent, or removing a trailing normative "is not antisemitic" clause). Each code is also assigned one of two templates based on whether its content is propositional/belief-based ("This text expresses: {}.") or describes an act or category ("This text is an example of: {}.").

I report results using the corrected hypotheses alongside the original version for comparison.

In [5]:
# Create dict of codes from nli_config, with grammatical hypothesis prefix

from nli_config import CODE_HYPOTHESES

codebook = {code_id: exemplar for code_id, (definition, exemplar) in CODE_HYPOTHESES.items()}

print(f"Parsed {len(codebook)} codes")
print(f"exemplar=0 (expresses): {sum(1 for e in codebook.values() if e == 0)}")
print(f"exemplar=1 (is an example of): {sum(1 for e in codebook.values() if e == 1)}")

127 codes defined
exemplar=0 (expresses): 44
exemplar=1 (is an example of): 83
Parsed 127 codes
exemplar=0 (expresses): 44
exemplar=1 (is an example of): 83


In [27]:
# small test, comparative v1 vs v2 using the hardcoded exemplar template per code, with LLM label
import pandas as pd

test_code = "N1BELIEFS"
test_definition, exemplar = CODE_HYPOTHESES[test_code]

TEMPLATES = {
    0: "This text expresses: {}.",
    1: "This text is an example of: {}.",
}
hypothesis_template = TEMPLATES[exemplar]
hypothesis = hypothesis_template.format(test_definition)

RUNS = [
    ("v1", nli_pipe),
    ("v2", nli_pipe_v2),
]

code_features = pd.read_csv("code_features.csv").set_index("comment_id")

def get_llm_label(comment_id, code_id):
    cols = {"E": f"{code_id}_E", "I": f"{code_id}_I", "A": f"{code_id}_A"}
    for label, col in cols.items():
        if code_features.loc[comment_id, col] == 1:
            return label
    return "N"

sample = judaism.sample(5, random_state=1)

for _, row in sample.iterrows():
    comment_id, text = row["comment_id"], row["text"]
    print(f"\nText: {text}")
    for model_label, pipe in RUNS:
        result = pipe(text, candidate_labels=[test_definition], hypothesis_template=hypothesis_template)
        print(f"  [{model_label}] {hypothesis}: {result['scores'][0]:.3f}")

    llm_label = get_llm_label(comment_id, test_code)
    print(f"  [LLM label] {test_code}: {llm_label}")


Text: YTA. The worst part of this is gall at being offended by his "unprofessional" behavior. I'm gonna shit all over this person and then be surprised that it upset them. 🤣🤣🤣 Fucking classic sociopathic reaction. It's only business until someone reacts negatively to me then it's personal and they were the ones in the wrong. You a garbage sub-human.
  [v1] This text expresses: anti-Jewish beliefs, attitudes, actions or systemic conditions.: 0.163
  [v2] This text expresses: anti-Jewish beliefs, attitudes, actions or systemic conditions.: 0.745
  [LLM label] N1BELIEFS: N

Text: When will Democrats condemn all of the vile, anti-Semitic, anti-American rhetoric that Ilhan Omar and the rest of her "Squad" consistently spews? 🤔
  [v1] This text expresses: anti-Jewish beliefs, attitudes, actions or systemic conditions.: 0.998
  [v2] This text expresses: anti-Jewish beliefs, attitudes, actions or systemic conditions.: 0.916
  [LLM label] N1BELIEFS: N

Text: I'm Jewish and a granddaughter of h

In [6]:
# Score full pilot corpus against all codes: v1 and v2, using the hardcoded exemplar template per code

import csv
import os
import torch
import pandas as pd
from tqdm.auto import tqdm

TEMPLATES = {
    0: "This text expresses: {}.",
    1: "This text is an example of: {}.",
}

RUNS = [
    ("v1_prefix", nli_pipe, 96),
    ("v2_prefix", nli_pipe_v2, 32),
]

texts = judaism["text"].tolist()
comment_ids = judaism["comment_id"].tolist()
code_ids = list(CODE_HYPOTHESES.keys())
expected_n = len(comment_ids)

for model_label, pipe, batch_size in RUNS:
    output_path = f"nli_zeroshot_scores_{model_label}.csv"

    already_done = set()
    if os.path.exists(output_path):
        existing = pd.read_csv(output_path)
        counts = existing["code_id"].value_counts()
        already_done = set(counts[counts >= expected_n].index)
        incomplete = set(counts[counts < expected_n].index)
        if incomplete:
            print(f"[{output_path}] Found {len(incomplete)} incomplete codes, will be re-scored: {sorted(incomplete)}")
            existing = existing[~existing["code_id"].isin(incomplete)]
            existing.to_csv(output_path, index=False)
        print(f"[{output_path}] Resuming: {len(already_done)} codes fully scored, skipping those")

    remaining = [(c, CODE_HYPOTHESES[c][0], CODE_HYPOTHESES[c][1]) for c in code_ids if c not in already_done]
    if not remaining:
        print(f"[{output_path}] Already complete, skipping run.")
        continue

    file_exists = os.path.exists(output_path)
    with open(output_path, "a", newline="") as f:
        writer = csv.writer(f)
        if not file_exists:
            writer.writerow(["comment_id", "code_id", "entailment_score"])

        pbar = tqdm(remaining, desc=f"Scoring ({model_label})", unit="code")
        for code_id, definition, exemplar in pbar:
            pbar.set_postfix(code=code_id)
            hypothesis_template = TEMPLATES[exemplar]

            with torch.no_grad(), torch.autocast(device_type="cuda", dtype=torch.float16):
                outputs = pipe(
                    texts,
                    candidate_labels=[definition],
                    hypothesis_template=hypothesis_template,
                    multi_label=True,
                    batch_size=batch_size,
                    truncation=True,
                )

            if isinstance(outputs, dict):
                outputs = [outputs]
            all_scores = [o["scores"][0] for o in outputs]

            if len(all_scores) != expected_n:
                print(f"WARNING: {code_id} produced {len(all_scores)} scores, expected {expected_n}. Skipping write.")
                continue

            for cid, score in zip(comment_ids, all_scores):
                writer.writerow([cid, code_id, score])
            f.flush()

    print(f"[{output_path}] All codes scored.")

[nli_zeroshot_scores_v1_prefix.csv] Resuming: 127 codes fully scored, skipping those
[nli_zeroshot_scores_v1_prefix.csv] Already complete, skipping run.
[nli_zeroshot_scores_v2_prefix.csv] Resuming: 106 codes fully scored, skipping those


Scoring (v2_prefix):   0%|          | 0/21 [00:00<?, ?code/s]

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


[nli_zeroshot_scores_v2_prefix.csv] All codes scored.
